In [ ]:
# Training untuk Huruf dan Angka
import pandas as pd
import numpy as np
import joblib
import os
import time
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier

# ==========================================================
# load CSV ke huruf
# ==========================================================
FILE_CSV  = "bisindo_alfabet_final.csv"  
MODEL_OUT = "model_huruf_indo2.pkl"          # Hasil akhir cetakan model

print("="*60)
print("  TRAINING RANDOM FOREST BISINDO (SINKRONISASI 92 FITUR)")
print("="*60)

if not os.path.exists(FILE_CSV):
    print(f"\n ERROR: File '{FILE_CSV}' tidak ditemukan!")
    exit()

# Load Data
df = pd.read_csv(FILE_CSV)
X = df.drop("label", axis=1).values
y = df["label"].values

print(f"Total data   : {len(df)} baris (Termasuk hasil Augmentasi)")
print(f"Jumlah kelas : {len(set(y))} huruf terdeteksi")
print(f"Jumlah fitur : {X.shape[1]} kolom (Pas 92 Fitur Teman)")

# Label Encoder
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Split Data (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_encoded
)

# Model Random Forest Murni
model = RandomForestClassifier(
    n_estimators=250, # Ditambah sedikit pohonnya agar makin pintar mencerna fitur jarak
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

print("\n Memulai proses training Random Forest... Harap tunggu...")
t0 = time.time()
model.fit(X_train, y_train)
print(f" -> Training selesai dalam {time.time() - t0:.1f} detik!")

# Evaluasi
pred = model.predict(X_test)
acc = accuracy_score(y_test, pred) * 100
print("\n" + "="*60)
print(f"  AKURASI RANDOM FOREST ON TEST SET : {acc:.2f}%")
print("="*60)

print("\nLaporan Klasifikasi Detail:")
print(classification_report(le.inverse_transform(y_test), le.inverse_transform(pred)))

# Simpan Paket Gabungan
saved_package = {
    'model': model,
    'label_encoder': le
}
joblib.dump(saved_package, MODEL_OUT)
print(f" ✅ Sukses! Paket Model RF 92 Fitur tersimpan di: {MODEL_OUT}")